# Projeto de Visualizacao da Informacao



Tema: filmes mais bem avaliados de 2025 usando dados reais da API do TMDB.



Este notebook e uma versao autocontida do projeto para entrega em um unico arquivo de codigo. Ele consome a API, limpa os dados e gera tres graficos relacionados ao mesmo dataset:



- Grafico de linha;

- Grafico de barras;

- Diagrama de cordas.

## 1. Importacoes



Principais bibliotecas usadas:



- `requests`: consumo da API;

- `pandas`: limpeza e organizacao do dataset;

- `matplotlib`: geracao dos graficos;

- `os` e `pathlib`: leitura de variaveis de ambiente e criacao de pastas.

In [1]:
from __future__ import annotations



import os

from pathlib import Path

from textwrap import wrap



import matplotlib.pyplot as plt

import pandas as pd

import requests

from matplotlib.path import Path as MatplotlibPath

from matplotlib.patches import Circle, PathPatch



OUTPUT_DIR = Path('output')

OUTPUT_DIR.mkdir(exist_ok=True)

SyntaxError: unexpected character after line continuation character (821802384.py, line 1)

## 2. Configuracao da API



O projeto usa a API do TMDB. Para autenticar, use um arquivo `.env` na mesma pasta do notebook contendo pelo menos uma das credenciais abaixo:



```text

TMDB_READ_TOKEN=seu_token_de_leitura

TMDB_API_KEY=sua_api_key

```



O token de leitura tem prioridade. Se ele nao existir, o codigo usa a API key.

In [ ]:
def load_dotenv(path: str | Path = '.env') -> None:

    dotenv_path = Path(path)

    if not dotenv_path.exists():

        return



    for line in dotenv_path.read_text(encoding='utf-8').splitlines():

        stripped_line = line.strip()

        if not stripped_line or stripped_line.startswith('#') or '=' not in stripped_line:

            continue



        key, value = stripped_line.split('=', 1)

        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))





load_dotenv()



TMDB_BASE_URL = 'https://api.themoviedb.org/3'

TMDB_LANGUAGE = 'pt-BR'

TMDB_REGION = 'BR'

PAGES = 10

RELEASE_YEAR = 2025

MIN_VOTE_COUNT = 100



TMDB_READ_TOKEN = os.getenv('TMDB_READ_TOKEN')

TMDB_API_KEY = os.getenv('TMDB_API_KEY')



if not TMDB_READ_TOKEN and not TMDB_API_KEY:

    raise RuntimeError('Configure TMDB_READ_TOKEN ou TMDB_API_KEY no arquivo .env.')

## 3. Consumo da API



O endpoint usado para buscar filmes e o `discover/movie`.



Filtros principais:



- `primary_release_year`: 2025;

- `primary_release_date.gte`: 2025-01-01;

- `primary_release_date.lte`: 2025-12-31;

- `sort_by`: vote_average.desc;

- `vote_count.gte`: 100.



Tambem e feita uma filtragem local depois do consumo, garantindo que o dataset final fique somente com filmes de 2025.

In [ ]:
def tmdb_get(path: str, params: dict[str, str | int | float | bool] | None = None) -> dict:

    request_params = dict(params or {})

    headers: dict[str, str] = {}



    if TMDB_READ_TOKEN:

        headers['Authorization'] = f'Bearer {TMDB_READ_TOKEN}'

    else:

        request_params['api_key'] = TMDB_API_KEY



    response = requests.get(

        f'{TMDB_BASE_URL}{path}',

        headers=headers,

        params=request_params,

        timeout=30,

    )

    response.raise_for_status()

    return response.json()





def fetch_movies() -> list[dict]:

    movies: list[dict] = []



    for page in range(1, PAGES + 1):

        payload = tmdb_get(

            '/discover/movie',

            {

                'language': TMDB_LANGUAGE,

                'region': TMDB_REGION,

                'page': page,

                'primary_release_year': RELEASE_YEAR,

                'primary_release_date.gte': f'{RELEASE_YEAR}-01-01',

                'primary_release_date.lte': f'{RELEASE_YEAR}-12-31',

                'sort_by': 'vote_average.desc',

                'include_adult': False,

                'include_video': False,

                'vote_count.gte': MIN_VOTE_COUNT,

            },

        )

        movies.extend(payload.get('results', []))



    return movies





def fetch_movie_genres() -> dict[int, str]:

    payload = tmdb_get('/genre/movie/list', {'language': TMDB_LANGUAGE})

    return {genre['id']: genre['name'] for genre in payload.get('genres', [])}





raw_movies = fetch_movies()

genre_names_by_id = fetch_movie_genres()



len(raw_movies), len(genre_names_by_id)

## 4. Transformacao do dataset



Nesta etapa, os dados brutos da API sao convertidos em um `DataFrame` do pandas.



Transformacoes realizadas:



- conversao de `release_date` para data;

- criacao de `release_year`;

- criacao de `release_month`;

- conversao de IDs de generos em nomes;

- criacao de `primary_genre`;

- conversao dos campos numericos;

- filtragem local para manter somente filmes de 2025.

In [ ]:
def genre_names(genre_ids: list[int] | float) -> list[str]:

    if not isinstance(genre_ids, list):

        return []

    return [genre_names_by_id.get(genre_id, f'Genero {genre_id}') for genre_id in genre_ids]





def primary_genre_name(genre_ids: list[int] | float) -> str:

    names = genre_names(genre_ids)

    if not names:

        return 'Sem genero'

    return names[0]





movies_df = pd.DataFrame(raw_movies)



if movies_df.empty:

    raise ValueError('A API do TMDB nao retornou filmes para o recorte informado.')



movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')

movies_df['release_year'] = movies_df['release_date'].dt.year

movies_df['release_month'] = movies_df['release_date'].dt.to_period('M').astype(str)

movies_df['genre_names'] = movies_df['genre_ids'].apply(genre_names)

movies_df['primary_genre'] = movies_df['genre_ids'].apply(primary_genre_name)

movies_df['popularity'] = pd.to_numeric(movies_df['popularity'], errors='coerce')

movies_df['vote_average'] = pd.to_numeric(movies_df['vote_average'], errors='coerce')

movies_df['vote_count'] = pd.to_numeric(movies_df['vote_count'], errors='coerce')



clean_df = movies_df.dropna(subset=['release_date', 'popularity']).copy()

clean_df = clean_df[clean_df['release_year'] == RELEASE_YEAR].copy()



clean_df[[

    'title',

    'release_date',

    'release_month',

    'vote_average',

    'vote_count',

    'original_language',

    'primary_genre',

]].head()

## 5. Grafico de linha



Este grafico mostra a nota media dos filmes por mes de lancamento em 2025.



A logica e:



- agrupar por `release_month`;

- calcular a media de `vote_average`;

- plotar a evolucao mensal.

In [ ]:
line_data = (

    clean_df.groupby('release_month', as_index=False)['vote_average']

    .mean()

    .sort_values('release_month')

)



figure, axis = plt.subplots(figsize=(11, 6))

axis.plot(line_data['release_month'], line_data['vote_average'], marker='o', linewidth=2)

axis.set_title('Nota media dos filmes mais bem avaliados de 2025 por mes')

axis.set_xlabel('Mes de lancamento')

axis.set_ylabel('Nota media')

axis.grid(True, alpha=0.25)

axis.tick_params(axis='x', rotation=35)

figure.tight_layout()



line_output = OUTPUT_DIR / 'tmdb_2025_avaliacao_linha.png'

figure.savefig(line_output, dpi=150)

plt.show()



line_output

## 6. Grafico de barras



Este grafico mostra os 15 filmes mais bem avaliados do recorte.



Como os titulos sao longos, as barras foram feitas na horizontal para melhorar a leitura.

In [ ]:
bar_data = (

    clean_df.groupby('title', as_index=False)['vote_average']

    .sum()

    .sort_values('vote_average', ascending=False)

    .head(15)

    .sort_values('vote_average', ascending=True)

)



figure, axis = plt.subplots(figsize=(12, 7))

bars = axis.barh(bar_data['title'], bar_data['vote_average'], color='#2f80ed')

axis.set_title('Filmes mais bem avaliados de 2025')

axis.set_xlabel('Nota media')

axis.set_ylabel('Filme')

axis.grid(axis='x', alpha=0.25)

axis.bar_label(bars, fmt='%.2f', padding=4, fontsize=8)

axis.margins(x=0.12)

figure.tight_layout()



bar_output = OUTPUT_DIR / 'tmdb_2025_avaliacao_barras.png'

figure.savefig(bar_output, dpi=150, bbox_inches='tight')

plt.show()



bar_output

## 7. Diagrama de cordas



Este grafico relaciona o idioma original do filme com o genero principal.



A espessura das conexoes representa o volume de votos (`vote_count`). Para manter a leitura clara, sao exibidas as 20 conexoes mais relevantes.

In [ ]:
def ordered_nodes(flows: pd.DataFrame, column: str, value_column: str) -> list[str]:

    totals = flows.groupby(column)[value_column].sum().sort_values(ascending=False)

    return [str(node) for node in totals.index]





def node_positions(nodes: list[str], x: float) -> dict[str, tuple[float, float]]:

    if len(nodes) == 1:

        return {nodes[0]: (x, 0.0)}

    step = 1.8 / (len(nodes) - 1)

    return {node: (x, 0.9 - index * step) for index, node in enumerate(nodes)}





def draw_nodes(axis, positions, align: str, color: str) -> None:

    text_offset = -0.05 if align == 'right' else 0.05

    for node, (x, y) in positions.items():

        axis.add_patch(Circle((x, y), radius=0.018, facecolor=color, edgecolor='white'))

        axis.text(x + text_offset, y, node, ha=align, va='center', fontsize=9)





def draw_chords(axis, flows, source_positions, target_positions, value_column: str) -> None:

    max_value = float(flows[value_column].max())

    for _, row in flows.iterrows():

        source = source_positions[str(row['original_language'])]

        target = target_positions[str(row['primary_genre'])]

        path = MatplotlibPath(

            [source, (-0.2, source[1]), (0.2, target[1]), target],

            [

                MatplotlibPath.MOVETO,

                MatplotlibPath.CURVE4,

                MatplotlibPath.CURVE4,

                MatplotlibPath.CURVE4,

            ],

        )

        linewidth = 0.75 + 6 * (float(row[value_column]) / max_value)

        axis.add_patch(

            PathPatch(

                path,

                facecolor='none',

                edgecolor='#9b51e0',

                linewidth=linewidth,

                alpha=0.35,

            )

        )





flows = (

    clean_df.groupby(['original_language', 'primary_genre'], as_index=False)['vote_count']

    .sum()

    .query('vote_count > 0')

    .sort_values('vote_count', ascending=False)

    .head(20)

)



sources = ordered_nodes(flows, 'original_language', 'vote_count')

targets = ordered_nodes(flows, 'primary_genre', 'vote_count')

source_positions = node_positions(sources, x=-0.85)

target_positions = node_positions(targets, x=0.85)



figure, axis = plt.subplots(figsize=(12, 7))

axis.set_title('Volume de avaliacoes por idioma e genero em 2025')

axis.axis('off')

axis.text(-0.85, 1.08, 'Idiomas', ha='center', va='bottom', fontsize=11, fontweight='bold')

axis.text(0.85, 1.08, 'Generos', ha='center', va='bottom', fontsize=11, fontweight='bold')

draw_nodes(axis, source_positions, align='right', color='#2f80ed')

draw_nodes(axis, target_positions, align='left', color='#27ae60')

draw_chords(axis, flows, source_positions, target_positions, 'vote_count')

axis.set_xlim(-1.45, 1.45)

axis.set_ylim(-1.15, 1.15)

figure.tight_layout()



chord_output = OUTPUT_DIR / 'tmdb_2025_avaliacao_cordas.png'

figure.savefig(chord_output, dpi=150, bbox_inches='tight')

plt.show()



chord_output

## 8. Resultado



Ao final da execucao, os tres arquivos de imagem sao gerados na pasta `output`:



- `tmdb_2025_avaliacao_linha.png`;

- `tmdb_2025_avaliacao_barras.png`;

- `tmdb_2025_avaliacao_cordas.png`.